# Multi-agent RAG demo

Choose one value in the project's `.env` and run the notebook:

- `LLM_PROVIDER=mock`: deterministic answers with real local MiniLM retrieval; no LLM server.
- `LLM_PROVIDER=ollama`: real local Qwen3 4B generation; start Ollama and pull `qwen3:4b` first.

API and notebook read the same configuration. No notebook switches are needed.
Native Python reads `.env` automatically; Docker Compose passes the same setting to its services.
Ollama defaults to `http://localhost:11434` natively and `http://host.docker.internal:11434`
in Docker. You only need `OLLAMA_URL` for a non-default installation.

After changing `.env`, use **Restart Kernel → Clear All Outputs → Run All**.
For Docker, also run `docker compose up -d api notebook` to refresh the container environment.
The first embedding download needs internet. An Ollama run takes several minutes on a laptop.

Each run creates fresh in-memory state. The controlled feedback, failure and 24-case baseline
experiments always use mock generation and are labelled as such. Section 10 verifies the
three answers from section 2 when Ollama is selected; it does not generate them a second time.

## 1. Inspect actual retrieval
Each short synthetic passage is one chunk. Inspect source IDs and raw cosine similarity.
Similarity measures relevance; confidence combines relevance, source authority and feedback.
Neither is a calibrated probability that a claim is true.

In [ ]:
import json
import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "rag_system").exists():
    root = root.parent
assert (root / "rag_system").exists(), "Launch from the repository or notebooks directory"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from rag_system.bootstrap import build_system
from rag_system.models import Feedback

# Same .env configuration as the API: LLM_PROVIDER=mock or LLM_PROVIDER=ollama.
system = build_system()
print(
    json.dumps(
        {
            "mode": system.llm.mode,
            "provider": system.llm.provider,
            "model": getattr(system.llm, "model", None),
            "ollama_url": getattr(system.llm, "base_url", None),
            "embeddings": system.store.embedder.name,
            "documents": len(system.store.documents),
        },
        indent=2,
    )
)

# Fail before the scenarios if the selected local LLM server is unavailable.
if system.llm.provider == "ollama":
    import httpx

    try:
        httpx.get(f"{system.llm.base_url}/api/tags", timeout=5).raise_for_status()
    except httpx.HTTPError as exc:
        raise RuntimeError(
            f"Cannot reach Ollama at {system.llm.base_url}. Start the Ollama app or "
            "run `ollama serve`, then rerun this cell. To use mock instead, set "
            "LLM_PROVIDER=mock in .env; with Docker run "
            "`docker compose up -d api notebook`, then restart the kernel."
        ) from exc

In [ ]:
# Texts come from knowledge.json; these vectors were computed at startup.
first_id = next(iter(system.store.vectors))
first_vector = system.store.vectors[first_id]
print("Documents:", len(system.store.documents))
print("Embedding dimensions:", first_vector.shape[0])
print("Example vector (first 8 values):", first_vector[:8])

hits = system.store.search("deploy microservice", "technical", top_k=3, min_similarity=0)
for hit in hits:
    print(hit.document.source_id, round(hit.similarity, 3), hit.document.title)
assert hits and all(h.document.domain == "technical" for h in hits)

## 2. Multi-domain examples
The planner creates domain-specific tasks. Domain and complexity control retrieval budgets.
Agents return claims with source IDs; the orchestrator resolves conflicts and synthesizes.
The mock planner uses explicit lexical rules; the live planner uses structured LLM output.

In [ ]:
queries = [
    "What’s the process for deploying a new microservice and what compliance checks are needed?",
    "How do I troubleshoot API performance issues while following our security policies?",
    "What business approvals are required for implementing a new data processing workflow?",
]
answers = []
for query in queries:
    answer = system.query(query)
    answers.append(answer)
    print("QUESTION:", query, "\nSTATUS:", answer.status)
    print("PLAN:", answer.plan.model_dump() if answer.plan else None)
    print("RETRIEVAL:", [(r.task.domain, r.top_k, len(r.retrieved)) for r in answer.results])
    print(answer.answer, "\n")
    assert answer.status == "answered" and answer.citations, {
        "provider": system.llm.provider,
        "error": answer.error,
        "domain_errors": {r.task.domain: r.error for r in answer.results if r.error},
    }

## 3. Resolve contradictory sources
An informal engineering note says **90 days**; the approved compliance policy says **30 days**.
We compare annotated facts with identical scope. Same-scope peers are checked beyond top-k,
so retrieval order cannot hide the policy. This is not arbitrary natural-language conflict detection.

In [ ]:
conflict_answer = system.query("How long should we retain production API request logs?")
print(conflict_answer.answer)
for conflict in conflict_answer.conflicts:
    print(conflict.model_dump())
assert any(c.selected_source_id == "comp-retention@v1#0" for c in conflict_answer.conflicts)
assert "90 days" not in conflict_answer.answer

## 4. Update knowledge and preserve historical citations
Updates require a higher version and replace the existing embedding. Idempotent retries are safe.
A previous answer retains its original document snapshot. All state resets with a new system.

In [ ]:
old = system.store.documents["comp-retention"]
updated = old.model_copy(
    update={"version": 2, "value": "14", "text": old.text.replace("30 days", "14 days")}
)
assert system.upsert(updated)
assert not system.upsert(updated)
after_update = system.query("How long should we retain production API request logs?")
print(after_update.answer)
assert any(e.document.source_id == "comp-retention@v2#0" for e in after_update.citations)
assert any(e.document.source_id == "comp-retention@v1#0" for e in conflict_answer.citations)

## 5. Simulated feedback changes retrieval ranking
Create two identical passages to isolate the effect from embedding similarity. A rating may
only target a cited current source, once per answer. Weights stay in [0.8, 1.2].
This is bounded reranking, not model training. Source authority remains the first conflict rule.

In [ ]:
from rag_system.llm import MockLLM
from rag_system.orchestrator import Orchestrator
from rag_system.vector_store import VectorStoreManager

original = system.store.documents["tech-deploy"]
copy = original.model_copy(update={"id": "aaa-deploy"})
feedback_demo = Orchestrator(VectorStoreManager([copy, original], system.store.embedder), MockLLM())
question = "How do I deploy a microservice?"


def ranking():
    return [
        (e.document.id, e.feedback_weight)
        for e in feedback_demo.store.search(question, "technical", 2, 0)
    ]


before = ranking()
rated = feedback_demo.query(question)
weight = feedback_demo.submit_feedback(
    Feedback(request_id=rated.request_id, source_id=original.source_id, helpful=True)
)
after = ranking()
print("Controlled MOCK feedback experiment:", before, "→", after)
assert before[0][0] == "aaa-deploy" and after[0][0] == "tech-deploy"
assert weight == 1.05

## 6. Unresolved conflicts and absent evidence
Equal-authority contradictions with similar scores are withheld. Unknown topics yield no evidence.
These controlled failure demonstrations use mock generation and the same actual embedding backend.

In [ ]:
policy = system.store.documents["comp-retention"]
alternative = policy.model_copy(
    update={"id": "retention-alt", "value": "60", "text": policy.text.replace("14 days", "60 days")}
)
conflict_demo = Orchestrator(
    VectorStoreManager([policy, alternative], system.store.embedder), MockLLM()
)
unresolved = conflict_demo.query("How long should we retain production API request logs?")
print(unresolved.answer)
assert unresolved.conflicts and unresolved.conflicts[0].selected_source_id is None
unknown = conflict_demo.query("Who won the World Cup?")
print(unknown.status, unknown.answer)
assert unknown.status == "no_evidence"

## 7. Partial failure is visible
A simulated compliance timeout must not turn into an apparently complete answer.

In [ ]:
from rag_system.llm import LLMError

failure_demo = build_system(
    provider="mock", embedding_backend=os.getenv("EMBEDDING_BACKEND", "fastembed")
)
original_generate = failure_demo.llm.generate


def fail_compliance(stage, instructions, payload, schema):
    if stage == "domain" and payload["task"]["domain"] == "compliance":
        raise LLMError("Simulated compliance timeout")
    return original_generate(stage, instructions, payload, schema)


failure_demo.llm.generate = fail_compliance
partial = failure_demo.query(queries[0])
print(partial.status, partial.answer)
print([(r.task.domain, r.error) for r in partial.results])
assert partial.status == "partial"

## 8. Deterministic retrieval baseline (mock)
24 hand-authored development cases, including the core examples and unanswerable questions.
This controlled baseline always uses mock generation, independently of the selected provider.
The report records raw retrieval recall separately from expected-source recall in the final answer.
A diagnostic pass is not proof of semantic support. Review real-model claims against their sources;
section 10 checks the three generated scenarios. These cases are not an independent holdout.

In [ ]:
from rag_system.evaluation import evaluate

# Keep this diagnostic baseline deterministic; real-model scenarios are checked below.
evaluation_system = build_system("mock")
report = evaluate(evaluation_system)
print(json.dumps({k: v for k, v in report.items() if k != "rows"}, indent=2))
print("Cases needing review:")
for row in report["rows"]:
    if not row["diagnostic_pass"]:
        print(row)
assert report["cases"] == 24

## 9. Performance monitoring and the OpenAPI boundary
Operational completion and explicit user feedback are separate metrics. HTTP 200 is not correctness.
This in-process HTTP check uses the same core. Run `uvicorn rag_system.api:app` for a standalone service.

In [ ]:
from fastapi.testclient import TestClient

from rag_system.api import create_app

print("Metrics and in-process API check for the main demo:", system.llm.mode, system.llm.provider)
print(json.dumps(system.metrics(), indent=2))
with TestClient(create_app(system)) as client:
    assert client.get("/health").status_code == 200
    assert "/query" in client.get("/openapi.json").json()["paths"]
    api_answer = client.post("/query", json={"text": queries[0]})
    assert api_answer.status_code == 200
    print("API:", api_answer.json()["status"], api_answer.json()["mode"])
    assert client.post("/query", json={"text": " "}).status_code == 422

## 10. Verify the three real-model scenarios

With `LLM_PROVIDER=ollama`, this checks the answers already generated in section 2.
With `LLM_PROVIDER=mock`, it prints an explicit skip message. No extra switch is required.

The checks cover successful completion, required domains, expected source IDs, non-empty
agent claims and real token usage. A stopped server is a failure, never a mock fallback.

Valid source IDs do not prove relevance or completeness; review the displayed answers
against their cited passages.


In [ ]:
if system.llm.provider == "mock":
    print("Real-model checks skipped: LLM_PROVIDER=mock. Select ollama in .env to run them.")
else:
    expected = [
        ({"technical", "compliance"}, {"tech-deploy", "comp-deploy"}),
        ({"technical", "compliance"}, {"tech-performance", "comp-debug"}),
        ({"business", "compliance"}, {"biz-approvals", "comp-data"}),
    ]
    for answer, (required_domains, required_sources) in zip(answers, expected, strict=True):
        assert answer.status == "answered", answer.error
        assert required_domains <= {task.domain for task in answer.plan.tasks}
        assert required_sources <= {source.document.id for source in answer.citations}
        assert all(result.claims and not result.error for result in answer.results)
        assert answer.input_tokens > 0 and answer.output_tokens > 0
        print("PASS:", [task.domain for task in answer.plan.tasks])
        print("TOKENS:", answer.input_tokens, "input /", answer.output_tokens, "output")
        print("TIME:", round(answer.timings_ms["total"] / 1000, 1), "seconds")
    print("All 3 local live scenarios passed.")

## Design choices and limitations
- Why exact in-memory search is sufficient for 18 short documents.
- What is simulated: mock generation/routing, annotated conflicts, feedback adaptation.
- What is real: embeddings (default), retrieval, citations, orchestration, API and updates.
- Why citation membership cannot prove entailment, and why similarity is not truth.
- When persistence, hybrid search, reranking and concurrency would be justified.
- State is process-local; notebook and API instances do not share updates.